In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

def scrape_imdb_top250():
    url = "https://www.imdb.com/chart/top/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }

    print(f"🔍 Fetching IMDB Top 250 from: {url}")
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # IMDB stores chart data in a JSON script tag
    script_tag = soup.find("script", {"id": "__NEXT_DATA__"})

    movies = []

    if script_tag:
        # Parse JSON embedded in the page
        data = json.loads(script_tag.string)
        chart_data = (
            data.get("props", {})
                .get("pageProps", {})
                .get("pageData", {})
                .get("chartTitles", {})
                .get("edges", [])
        )

        for rank, edge in enumerate(chart_data, start=1):
            node = edge.get("node", {})
            title = node.get("titleText", {}).get("text", "N/A")
            year = node.get("releaseYear", {}).get("year", "N/A")
            rating = node.get("ratingsSummary", {}).get("aggregateRating", "N/A")
            votes = node.get("ratingsSummary", {}).get("voteCount", "N/A")
            runtime_seconds = node.get("runtime", {}) or {}
            runtime_min = runtime_seconds.get("seconds", 0) // 60 if runtime_seconds else "N/A"

            movies.append({
                "Rank": rank,
                "Title": title,
                "Year": year,
                "Rating": rating,
                "Votes": votes,
                "Runtime (min)": runtime_min,
            })

    else:
        # Fallback: scrape HTML directly
        print("⚠️  JSON data not found, falling back to HTML parsing...")
        rows = soup.select("li.ipc-metadata-list-summary-item")

        for rank, row in enumerate(rows, start=1):
            title_tag = row.select_one("h3.ipc-title__text")
            rating_tag = row.select_one("span.ipc-rating-star--imdb")
            meta_tags = row.select("span.cli-title-metadata-item")

            title = title_tag.get_text(strip=True).split(". ", 1)[-1] if title_tag else "N/A"
            rating = rating_tag.get_text(strip=True).split()[0] if rating_tag else "N/A"
            year = meta_tags[0].get_text(strip=True) if len(meta_tags) > 0 else "N/A"
            runtime = meta_tags[1].get_text(strip=True) if len(meta_tags) > 1 else "N/A"

            movies.append({
                "Rank": rank,
                "Title": title,
                "Year": year,
                "Rating": rating,
                "Runtime": runtime,
            })

    if not movies:
        print("❌ No data found. IMDB may have changed their page structure.")
        return

    # Save to CSV
    df = pd.DataFrame(movies)
    output_file = "imdb_top250.csv"
    df.to_csv(output_file, index=False, encoding="utf-8")

    print(f"\n✅ Scraped {len(df)} movies successfully!")
    print(f"📁 Saved to: {output_file}")
    print("\n📊 Preview (Top 10):")
    print(df.head(10).to_string(index=False))


if __name__ == "__main__":
    scrape_imdb_top250()

🔍 Fetching IMDB Top 250 from: https://www.imdb.com/chart/top/

✅ Scraped 250 movies successfully!
📁 Saved to: imdb_top250.csv

📊 Preview (Top 10):
 Rank                                             Title  Year  Rating   Votes  Runtime (min)
    1                          The Shawshank Redemption  1994     9.3 3165758            142
    2                                     The Godfather  1972     9.2 2211069            175
    3                                   The Dark Knight  2008     9.1 3143231            152
    4                             The Godfather Part II  1974     9.0 1485684            202
    5                                      12 Angry Men  1957     9.0  976583             96
    6     The Lord of the Rings: The Return of the King  2003     9.0 2150818            201
    7                                  Schindler's List  1993     9.0 1576693            195
    8 The Lord of the Rings: The Fellowship of the Ring  2001     8.9 2191882            178
    9           

In [3]:
import os
os.getcwd()

'D:\\'

In [7]:
pip install requests beautifulsoup4 pandas mysql-connector-python

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/16.5 MB 10.1 MB/s eta 0:00:02
   -------- ------------------------------- 3.7/16.5 MB 9.5 MB/s eta 0:00:02
   ------------- -------------------------- 5.8/16.5 MB 9.8 MB/s eta 0:00:02
   ------------------- -------------------- 8.1/16.5 MB 9.7 MB/s eta 0:00:01
   ------------------------ --------------- 10.2/16.5 MB 9.7 MB/s eta 0:00:01
   ----------------------------- ---------- 12.1/16.5 MB 9.7 MB/s eta 0:00:01
   ---------------------------------- ----- 14.2/16.5 MB 9.6 MB/s eta 0:00:01
   -------------------------------------- - 15.7/16.5 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.5 MB 9.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pyodbc

Note: you may need to restart the kernel to use updated packages.


## PACKAGES

In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import sqlite3
import pyodbc  

In [18]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [20]:
# ─────────────────────────────────────────────
# 🔧 CONFIGURE YOUR SQL SERVER CONNECTION HERE
# ─────────────────────────────────────────────
DB_CONFIG = {
    "server":   r"VEATH\SQLEXPRESS",          # ✅ raw string (fixes \S warning)
    "database": "imdb_db",
    "driver":   "{ODBC Driver 17 for SQL Server}",
}

# ─────────────────────────────────────────────
# 📁 OUTPUT FILE NAMES
# ─────────────────────────────────────────────
CSV_FILE    = "imdb_top250.csv"
SQLITE_FILE = "imdb_top250.db"


# ══════════════════════════════════════════════
# 🔍 SCRAPER
# ══════════════════════════════════════════════
def scrape_imdb_top250():
    url = "https://www.imdb.com/chart/top/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }

    print(f"🔍 Fetching IMDB Top 250 from: {url}")
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    script_tag = soup.find("script", {"id": "__NEXT_DATA__"})

    movies = []

    if script_tag:
        data = json.loads(script_tag.string)
        chart_data = (
            data.get("props", {})
                .get("pageProps", {})
                .get("pageData", {})
                .get("chartTitles", {})
                .get("edges", [])
        )

        for rank, edge in enumerate(chart_data, start=1):
            node           = edge.get("node", {})
            title          = node.get("titleText", {}).get("text", "N/A")
            year           = node.get("releaseYear", {}).get("year", "N/A")
            rating         = node.get("ratingsSummary", {}).get("aggregateRating", "N/A")
            votes          = node.get("ratingsSummary", {}).get("voteCount", "N/A")
            runtime_data   = node.get("runtime", {}) or {}
            runtime_min    = runtime_data.get("seconds", 0) // 60 if runtime_data else "N/A"

            movies.append({
                "Rank":          rank,
                "Title":         title,
                "Year":          year,
                "Rating":        rating,
                "Votes":         votes,
                "Runtime (min)": runtime_min,
            })

    else:
        print("⚠️  JSON not found, falling back to HTML parsing...")
        rows = soup.select("li.ipc-metadata-list-summary-item")

        for rank, row in enumerate(rows, start=1):
            title_tag  = row.select_one("h3.ipc-title__text")
            rating_tag = row.select_one("span.ipc-rating-star--imdb")
            meta_tags  = row.select("span.cli-title-metadata-item")

            title   = title_tag.get_text(strip=True).split(". ", 1)[-1] if title_tag else "N/A"
            rating  = rating_tag.get_text(strip=True).split()[0] if rating_tag else "N/A"
            year    = meta_tags[0].get_text(strip=True) if len(meta_tags) > 0 else "N/A"
            runtime = meta_tags[1].get_text(strip=True) if len(meta_tags) > 1 else "N/A"

            movies.append({
                "Rank":          rank,
                "Title":         title,
                "Year":          year,
                "Rating":        rating,
                "Votes":         "N/A",
                "Runtime (min)": runtime,
            })

    return movies


# ══════════════════════════════════════════════
# 💾 SAVE TO CSV
# ══════════════════════════════════════════════
def save_to_csv(df):
    df.to_csv(CSV_FILE, index=False, encoding="utf-8")
    print(f"✅ CSV saved  →  {CSV_FILE}")


# ══════════════════════════════════════════════
# 🗄️  SAVE TO SQLITE
# ══════════════════════════════════════════════
def save_to_sqlite(df):
    conn = sqlite3.connect(SQLITE_FILE)
    df.to_sql("imdb_top250", conn, if_exists="replace", index=False)
    conn.close()
    print(f"✅ SQLite DB saved  →  {SQLITE_FILE}  (table: imdb_top250)")


# ══════════════════════════════════════════════
# 🛢️  SAVE TO SQL SERVER (Windows Authentication)
# ══════════════════════════════════════════════
def save_to_mysql(movies):
    try:
        # ✅ Windows Authentication connection string (no username/password needed)
        conn_str = (
            f"DRIVER={DB_CONFIG['driver']};"
            f"SERVER={DB_CONFIG['server']};"
            f"DATABASE={DB_CONFIG['database']};"
            f"Trusted_Connection=yes;"        # ✅ This enables Windows Auth
        )
        conn = pyodbc.connect(conn_str)
        print("✅ Connected to SQL Server.")
        cursor = conn.cursor()

        # Create table if not exists
        cursor.execute("""
            IF NOT EXISTS (
                SELECT * FROM sysobjects WHERE name='imdb_top250' AND xtype='U'
            )
            CREATE TABLE imdb_top250 (
                id          INT IDENTITY(1,1) PRIMARY KEY,
                rank_no     INT,
                title       NVARCHAR(255),
                year        INT,
                rating      FLOAT,
                votes       BIGINT,
                runtime_min INT,
                created_at  DATETIME DEFAULT GETDATE()
            )
        """)

        # Clear old data
        cursor.execute("DELETE FROM imdb_top250")

        # Insert records
        insert_query = """
            INSERT INTO imdb_top250 (rank_no, title, year, rating, votes, runtime_min)
            VALUES (?, ?, ?, ?, ?, ?)
        """  # ✅ SQL Server uses ? placeholders, not %s

        records = [
            (
                m["Rank"],
                m["Title"],
                m["Year"]          if str(m["Year"]).isdigit()         else None,
                m["Rating"]        if m["Rating"]        != "N/A"      else None,
                m["Votes"]         if m["Votes"]         != "N/A"      else None,
                m["Runtime (min)"] if m["Runtime (min)"] != "N/A"      else None,
            )
            for m in movies
        ]
        cursor.executemany(insert_query, records)
        conn.commit()

        print(f"✅ SQL Server saved  →  {len(records)} rows inserted into `imdb_top250`")
        cursor.close()
        conn.close()
        print("🔒 SQL Server connection closed.")

    except pyodbc.Error as e:
        print(f"❌ SQL Server Error: {e}")


# ══════════════════════════════════════════════
# 🚀 MAIN
# ══════════════════════════════════════════════
def main():
    print("=" * 55)
    print("        🎬 IMDB Top 250 Scraper")
    print("=" * 55)

    # Step 1: Scrape
    movies = scrape_imdb_top250()
    if not movies:
        print("❌ No data scraped. Exiting.")
        return

    df = pd.DataFrame(movies)

    # Step 2: Save to CSV
    print("\n📄 Saving to CSV...")
    save_to_csv(df)

    # Step 3: Save to SQLite
    print("\n🗄️  Saving to SQLite DB...")
    save_to_sqlite(df)

    # Step 4: Save to SQL Server
    print("\n🛢️  Saving to SQL Server...")
    save_to_mysql(movies)

    # Step 5: Preview
    print(f"\n✅ Total movies scraped: {len(df)}")
    print("\n📊 Preview (Top 10):")
    print(df.head(10).to_string(index=False))
    print("=" * 55)


if __name__ == "__main__":
    main()

        🎬 IMDB Top 250 Scraper
🔍 Fetching IMDB Top 250 from: https://www.imdb.com/chart/top/

📄 Saving to CSV...
✅ CSV saved  →  imdb_top250.csv

🗄️  Saving to SQLite DB...
✅ SQLite DB saved  →  imdb_top250.db  (table: imdb_top250)

🛢️  Saving to SQL Server...
✅ Connected to SQL Server.
✅ SQL Server saved  →  250 rows inserted into `imdb_top250`
🔒 SQL Server connection closed.

✅ Total movies scraped: 250

📊 Preview (Top 10):
 Rank                                             Title  Year  Rating   Votes  Runtime (min)
    1                          The Shawshank Redemption  1994     9.3 3167580            142
    2                                     The Godfather  1972     9.2 2212543            175
    3                                   The Dark Knight  2008     9.1 3145194            152
    4                             The Godfather Part II  1974     9.0 1486653            202
    5                                      12 Angry Men  1957     9.0  977228             96
    6     Th